# Orthanc Study Metadata Test
 
Use this notebook to test custom study-level metadata fields on Orthanc: 
- `groundTruthReport` 
- `mammoReport` 
- `medGemmaReport` 
- `false_report_finding`
- `missing_finding`
- `mischaracterization_finding`
- `misidentification_finding`
- `incorrect_birads_assessment_finding`
- `breast_denstity_mismatch`

## 1) Configuration 
 
Set these env vars before launching Jupyter if needed: 
- `ORTHANC_BASE_URL` (default: `http://localhost:8042/pacs`) 
- `ORTHANC_USERNAME` (optional) 
- `ORTHANC_PASSWORD` (optional) 
 
If your server does not use the `/pacs` prefix, set `ORTHANC_BASE_URL` accordingly (for example `http://localhost:8042`).

In [5]:
import os
import json
import base64
import urllib.parse
import urllib.request
import urllib.error

ORTHANC_BASE_URL = os.getenv('ORTHANC_BASE_URL', 'http://localhost:8042').rstrip('/')
ORTHANC_USERNAME = os.getenv('ORTHANC_USERNAME')
ORTHANC_PASSWORD = os.getenv('ORTHANC_PASSWORD')

print('ORTHANC_BASE_URL =', ORTHANC_BASE_URL)
print('Auth configured =', bool(ORTHANC_USERNAME and ORTHANC_PASSWORD))

ORTHANC_BASE_URL = http://localhost:8042
Auth configured = False


In [6]:
def _auth_header():
    if not (ORTHANC_USERNAME and ORTHANC_PASSWORD):
        return None
    token = base64.b64encode(f"{ORTHANC_USERNAME}:{ORTHANC_PASSWORD}".encode('utf-8')).decode('utf-8')
    return f"Basic {token}"

def _request(method, path, *, params=None, body=None, content_type=None, timeout=30):
    query = f"?{urllib.parse.urlencode(params)}" if params else ''
    url = f"{ORTHANC_BASE_URL}{path}{query}"
    headers = {}
    auth = _auth_header()
    if auth:
        headers['Authorization'] = auth
    if content_type:
        headers['Content-Type'] = content_type

    data = None
    if body is not None:
        data = body if isinstance(body, bytes) else str(body).encode('utf-8')

    req = urllib.request.Request(url=url, data=data, headers=headers, method=method)
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        payload = resp.read()
        return {
            'status': resp.status,
            'headers': dict(resp.headers),
            'bytes': payload,
            'text': payload.decode('utf-8', errors='replace')
        }

def orthanc_get(path, params=None):
    return _request('GET', path, params=params)

def orthanc_put_text(path, value):
    return _request('PUT', path, body=value, content_type='text/plain')

def orthanc_delete(path):
    return _request('DELETE', path)

## 2) Find Orthanc Study ID from StudyInstanceUID

In [7]:
def get_studies_with_uid():
    params = {'expand': 1, 'requestedTags': 'StudyInstanceUID'}
    r = orthanc_get('/studies', params=params)
    return json.loads(r['text'])

def get_orthanc_study_id(study_instance_uid):
    studies = get_studies_with_uid()
    for s in studies:
        uid = (s.get('RequestedTags') or {}).get('StudyInstanceUID')
        if uid == study_instance_uid:
            return s.get('ID')
    return None

In [ ]:
# Set your StudyInstanceUID here
# study_instance_uid = 'PUT_YOUR_STUDY_INSTANCE_UID_HERE'
study_instance_uid = '1.2.840.113654.2.70.1.213012051129742288091890829040441839772'

orthanc_study_id = get_orthanc_study_id(study_instance_uid)
print('StudyInstanceUID:', study_instance_uid)
print('Orthanc study ID:', orthanc_study_id)
if not orthanc_study_id:
    raise ValueError('Study not found. Check StudyInstanceUID and ORTHANC_BASE_URL.')

## 2.5) Read in Spreadsheet

In [9]:
import pandas as pd

df = pd.read_csv('/Users/katelynmorrison/Downloads/upmc_joined_zs_mammo_clip_medgemma.csv')
df_llm_judge_report = pd.read_csv('/Users/katelynmorrison/Downloads/upmc_llm_judge_report.csv')

# df = pd.read_csv('/path/to/folder/upmc_joined_zs_mammo_clip_medgemma.csv')
# df_llm_judge_report = pd.read_csv('/path/to/folder/upmc_llm_judge_report.csv')

data = {}

# Loop rows
for _, row in df.iterrows():
    patient_id_csv = row['patient_id']
    truth = row['ground_truth_report']
    mammo = row['final_generated_report_zs_mammo_clip']
    gemma = row['final_generated_report_zs_medgemma']
    data[patient_id_csv] = {
        'ground_truth_report': truth,
        'final_generated_report_zs_mammo_clip': mammo,
        'final_generated_report_zs_medgemma': gemma
    }

for _, row in df_llm_judge_report.iterrows():
    patient_id_csv = row['patient_id']
    false_report_finding = row['sig_a_false_report']
    missing_finding = row['sig_b_missing_finding']
    mischaracterization_finding = row['sig_c_mischaracterization']
    misidentification_finding = row['sig_d_location_laterality']
    incorrect_birads_assessment_finding = row['sig_e_incorrect_birads']
    breast_denstity_mismatch = row['sig_f_density_mismatch']
    sig_a_false_report_expl = row['sig_a_false_report_expl']
    sig_b_missing_finding_expl = row['sig_b_missing_finding_expl']
    sig_c_mischaracterization_expl = row['sig_c_mischaracterization_expl']
    sig_d_location_laterality_expl = row['sig_d_location_laterality_expl']
    sig_e_incorrect_birads_expl = row['sig_e_incorrect_birads_expl']
    sig_f_density_mismatch_expl = row['sig_f_density_mismatch_expl']

    if patient_id_csv not in data:
        data[patient_id_csv] = {}

    data[patient_id_csv].update({
        'false_report_finding': false_report_finding,
        'missing_finding': missing_finding,
        'mischaracterization_finding': mischaracterization_finding,
        'misidentification_finding': misidentification_finding,
        'incorrect_birads_assessment_finding': incorrect_birads_assessment_finding,
        'breast_denstity_mismatch': breast_denstity_mismatch,
        'false_report_finding_explanation': sig_a_false_report_expl,
        'missing_finding_explanation': sig_b_missing_finding_expl,
        'mischaracterization_finding_explanation': sig_c_mischaracterization_expl,
        'misidentification_finding_explanation': sig_d_location_laterality_expl,
        'incorrect_birads_assessment_finding_explanation': sig_e_incorrect_birads_expl,
        'breast_denstity_mismatch_explanation': sig_f_density_mismatch_expl
    })

In [ ]:
studies = json.loads(orthanc_get(f'/studies/')['text'])

for id in studies:
    study = json.loads(orthanc_get(f'/studies/{id}')['text'])

    # 1) try study-level fields first
    patient_id = (
        (study.get('PatientMainDicomTags') or {}).get('PatientID')
        or (study.get('MainDicomTags') or {}).get('PatientID')
    )

    # 2) fallback to parent patient resource
    if not patient_id:
        parent_patient_id = study.get('ParentPatient')
        if parent_patient_id:
            patient = json.loads(orthanc_get(f'/patients/{parent_patient_id}')['text'])
            patient_id = (patient.get('MainDicomTags') or {}).get('PatientID')

    print('PatientID:', patient_id)

    metadata_payload = {
        'groundTruthReport': data[int(patient_id)]['ground_truth_report'],
        'mammoReport': data[int(patient_id)]['final_generated_report_zs_mammo_clip'],
        'medGemmaReport': data[int(patient_id)]['final_generated_report_zs_medgemma'],
        'false_report_finding': data[int(patient_id)]['false_report_finding'],
        'missing_finding': data[int(patient_id)]['missing_finding'],
        'mischaracterization_finding': data[int(patient_id)]['mischaracterization_finding'],
        'misidentification_finding': data[int(patient_id)]['misidentification_finding'],
        'incorrect_birads_assessment_finding': data[int(patient_id)]['incorrect_birads_assessment_finding'],
        'breast_denstity_mismatch': data[int(patient_id)]['breast_denstity_mismatch'],
        'false_report_finding_explanation': data[int(patient_id)]['false_report_finding_explanation'],
        'missing_finding_explanation': data[int(patient_id)]['missing_finding_explanation'],
        'mischaracterization_finding_explanation': data[int(patient_id)]['mischaracterization_finding_explanation'],
        'misidentification_finding_explanation': data[int(patient_id)]['misidentification_finding_explanation'],
        'incorrect_birads_assessment_finding_explanation': data[int(patient_id)]['incorrect_birads_assessment_finding_explanation'],
        'breast_denstity_mismatch_explanation': data[int(patient_id)]['breast_denstity_mismatch_explanation']
    }

    for key, value in metadata_payload.items():
        orthanc_put_text(f'/studies/{orthanc_study_id}/metadata/{key}', value)
        print(f'Wrote metadata key: {key}')


## 4) Read Back Metadata

In [ ]:
for key in ['groundTruthReport', 'mammoReport', 'medGemmaReport']:
    try:
        r = orthanc_get(f'/studies/{orthanc_study_id}/metadata/{key}')
        print(key, r['text'])
    except Exception as e:
        print(key, '(missing or error)', e)


## 5) Optional Cleanup (Delete Metadata Keys) 
 
Uncomment and run if you want to remove test metadata values.

In [ ]:
# Example
# for key in [1024,1025,1026,1027,1028,1029,1030,1031,1032,1033,1034,1035,1036,1037,1038]:  # replace with actual keys you want to delete
#     orthanc_delete(f'/studies/{orthanc_study_id}/metadata/{key}')
#     print(f'Deleted metadata key: {key}')